# Tesla Finanzanalyse #

## Installieren der Bibliotheken yfinance, pandas, matplotlib und seaborn ##

In [1]:
pip install yfinance pandas matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


## Importieren verschiedener Bibliotheken ##

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
from datetime import datetime
from requests.exceptions import RequestException

## Funktionsdefinition zum Holen von Aktiendaten von Yahoo Finance & zum Bereinigen und Bearbeiten dieser ##


In [3]:
def get_stock_data(ticker_symbol, start, end):
    """
    Lädt historische Finanzdaten von Yahoo Finance.

    Parameters:
    -----------
    ticker_symbol : str
        Das eindeutige Börsenkürzel der Aktie (z.B. 'AAPL', 'META').
    start : str
        Das Startdatum des Analysezeitraums im ISO-Format 'YYYY-MM-DD'.
    end : str
        Das Enddatum des Analysezeitraums im ISO-Format 'YYYY-MM-DD'.
    """
    print(f"Lade Daten für {ticker_symbol} von {start} bis {end}...")
    
    try:
        # Daten mit yfinance herunterladen
        stock_data = yf.download(tickers=ticker_symbol, start=start, end=end)
        
        # Prüfen, ob die Tabelle leer ist (z.B. wegen Tippfehler im Ticker)
        if stock_data.empty:
            raise ValueError(f"Keine Daten gefunden für das Kürzel '{ticker_symbol}'. Bitte überprüfe die Eingabe.")
   
    except ValueError as val_err:
        # Hier wird die eigens geworfene Exception bei leeren Daten abgefangen
        print(f"Validierungsfehler: {val_err}")
        return None
        
    except Exception as e:
        # Hier werden alle unerwarteten, technischen Fehler abgefangen (z.B. kein Internet)
        print(f"Technischer Fehler beim Datenabruf: {e}")
        return None

    return stock_data

def clean_stock_data(stock_data):
    """
    Bereinigt die Daten.
    
    Parameter:
    -----------
    stock_data : str
        Die von yfinance ausgegebenen Börsendaten.
    """

    # Daten bereinigen, indem fehlende Werte gestrichen werden
    stock_data = stock_data.dropna()
        
    # Index zurücksetzen, um das Datum als Spalte einzubeziehen (ohne Erstellen einer Kopie der Tabelle)
    stock_data.reset_index(inplace=True)
        
    # Zeitzonen-Informationen entfernen, falls vorhanden (wenn es 'Date' in Stock_data gibt und das Datum als Datum & nicht z.B. Text formatiert ist und an diesem Datum eine Zeitzone hängt: Entfernen der Zeitzone)
    if 'Date' in stock_data.columns and pd.api.types.is_datetime64_any_dtype(stock_data['Date']) and stock_data['Date'].dt.tz is not None:
        stock_data['Date'] = stock_data['Date'].dt.tz_localize(None)

    # Die Spaltenstruktur anpassen (da die Spaltenstruktur von yfinance als Tupel übernommen wird) --> Macht aus der Spalte ('Close', 'TSLA') einfach 'Close' (wichtig, da ich 'Close' in der nächsten Funktionsdefinition ansprechen möchte)
    stock_data.columns = [col[0] if isinstance(col, tuple) else col for col in stock_data.columns]
        
    # Nur die Spalten behalten, die benötigt werden
    required_columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
    existing_columns = [col for col in required_columns if col in stock_data.columns]
        
    print("\nDaten erfolgreich geladen und bereinigt!\n")
    return stock_data[existing_columns]

## Definieren der Kennzahlen-Funktion ##
Kennzahlen-Funktion für XXX

In [4]:
def calculate_metrics(stock_data):
    """
    Berechnet die täglichen Renditen sowie die innertägige Volatilität 
    basierend auf den Kursspalten.
    """
    print("\nBerechne tägliche Renditen und absolute Gewinne/Verluste...")
    
    try:
        required_cols = ['Close', 'High', 'Low', 'Open']
        if not all(col in stock_data.columns for col in required_cols):
            raise KeyError("Erforderliche Spalten ('Close', 'High', 'Low' oder 'Open') fehlen in den Daten.")
            
        # Berechnung der täglichen Rendite über die 'Close'-Spalte (täglicher Schlusskurs): pct_change --> Rendite = (Abschlusskurs Heute - Abschlusskurs Gestern) geteilt durch Abschlusskurs Gestern
        stock_data['Daily_Return'] = stock_data['Close'].pct_change()

        # Berechnung absoluter Gewinn/Verlust in USD zum Vortag
        stock_data['Daily_Gain_Loss_USD'] = stock_data['Close'].diff()

        print("\nRenditen und innertägige Kennzahlen erfolgreich berechnet!")
        return stock_data
        
    except KeyError as key_err:
        print(f" Fehler in den Datenstrukturen: {key_err}")
        return None
        
    except TypeError as type_err:
        print(f" Datentyp-Fehler bei der Berechnung: {type_err}")
        return None
        

## Definieren der erweiterten Analyse-Funktion ##

Funktion für XXX

In [5]:
def calculate_advanced_metrics(stock_data):
    """
    Analyse-Funktion für fortgeschrittene Metriken:
    - Risiko & Volatilität (Rollierende 21d-Vola & Intraday-Handelsspanne)
    - Trends (SMA 50 & SMA 200)
    """
    print("\nStarte erweiterte Finanzanalyse...")

   # Prüfung, ob alle Basis-Spalten aus den Phasen davor da sind
    try:
        required_advanced_cols = ['Daily_Return', 'Close', 'Volume', 'High', 'Low', 'Open']
        if not all(col in stock_data.columns for col in required_advanced_cols):
            raise KeyError("Erforderliche Spalten für die erweiterte Analyse fehlen.")

        # Innertägige Volatilität als prozentuale Handelsspanne (zeigt das Schwankungsrisiko innerhalb desselben Handelstages)
        stock_data['Intraday_Volatility_Pct'] = (stock_data['High'] - stock_data['Low']) / stock_data['Open']
        
        # Berechnung der rollierenden 21-Tage-Volatilität (Standardabweichung * Wurzel aus 252 zur Annualisierung; 21 Handelstage entsprechen ca. einem Kalendermonat)
        
        # .std() berechnet die Standardabweichung im Fenster, np.sqrt(252) rechnet es aufs Jahr hoch
        stock_data['Rolling_Volatility_21d'] = stock_data['Daily_Return'].rolling(window=21).std() * np.sqrt(252)

        # Trends: 50-Tage und 200-Tage gleitender Durchschnitt (SMA)
        stock_data['SMA_50'] = stock_data['Close'].rolling(window=50).mean()
        stock_data['SMA_200'] = stock_data['Close'].rolling(window=200).mean()

        print("\nErweiterte Finanzanalyse erfolgreich abgeschlossen!")
        return stock_data

    except Exception as e:
        print(f"Fehler bei der erweiterten Finanzanalyse: {e}")
        return None

## Definieren der Funktionen zur Datenvisualisierung ##

## Funktion zum Plotten des Kursverlaufs ##
Funktion XXX

In [6]:
def plot_price_history(stock_data, stock_symbol):
    """
    Visualisiert den reinen historischen Schlusskursverlauf der Aktie.
    """
    print(f"\nGeneriere reinen Kursverlauf-Plot für {stock_symbol}...")
    
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.figure(figsize=(12, 6))
    
    plt.plot(stock_data['Date'], stock_data['Close'], color='#1f77b4', linewidth=1.5, label='Schlusskurs (Close)')
    
    # Statt der 6 Zeilen Code schreibst du jetzt nur noch:
    apply_standard_chart_styling(
        title=f"Historischer Kursverlauf ({stock_symbol})", 
        ylabel="Kurs in USD", 
        show_grid=True
    )
    
    plt.savefig(f"{stock_symbol}_01_kursverlauf.png", dpi=300)
    plt.close()
    print("Kursverlauf-Diagramm separat gespeichert.")

## Funktion zum Plotten der täglichen Renditen ##
Funktion XXX

In [7]:
def plot_daily_returns(stock_data, stock_symbol):
    """
    Visualisiert die täglichen prozentualen Renditen zur Analyse der Volatilitätsdynamik.
    """
    print(f"\nGeneriere Renditedynamik-Plot für {stock_symbol}...")
    
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.figure(figsize=(12, 6))
    
    # Renditen in Prozent umrechnen
    daily_return_pct = stock_data['Daily_Return'] * 100
    
    # Plot als feine Nadeln/Linien
    plt.plot(stock_data['Date'], daily_return_pct, color='#ff7f0e', alpha=0.6, linewidth=0.7, label='Tägliche Rendite')
    
    # Eine rote Nulllinie zur Orientierung einziehen
    plt.axhline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.6)

    # Hier wird die Y-Achse angepasst:
    apply_standard_chart_styling(
        title=f"Tägliche Renditedynamik ({stock_symbol})", 
        ylabel="Rendite in %", 
        show_grid=True
    )
    
    plt.savefig(f"{stock_symbol}_02_taegliche_renditen.png", dpi=300)
    plt.close()
    print("Renditen-Diagramm separat gespeichert.")

### Funktion zum Plotten von Kurs und Trends ###
Funktion XXX

In [8]:
def plot_price_trends(stock_data, stock_symbol):
    """
    Visualisiert den Schlusskurs zusammen mit dem 50-Tage- und 200-Tage-Trend.
    """
    print(f"\nGeneriere Kurs- und Trend-Diagramm für {stock_symbol}...")
    
    # Stil der Grafik auswählen
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.figure(figsize=(12, 6))
    
    # 1. Schlusskurs plotten (Hauptlinie)
    plt.plot(stock_data['Date'], stock_data['Close'], label='Schlusskurs (Close)', color='#1f77b4', linewidth=1.5)
    
    # 2. Trends einzeichnen (Geglättete Linien)
    plt.plot(stock_data['Date'], stock_data['SMA_50'], label='50-Tage-Trend (SMA 50)', color='#ff7f0e', linestyle='--', linewidth=1.2)
    plt.plot(stock_data['Date'], stock_data['SMA_200'], label='200-Tage-Trend (SMA 200)', color='#2ca02c', linestyle='-.', linewidth=1.2)

    # Hier hattest du kein Grid aktiv:
    apply_standard_chart_styling(
        title=f"Kursverlauf und Trendanalysen ({stock_symbol})", 
        ylabel="Kurs in USD"
    )
    
    # Grafik speichern
    plt.savefig(f"{stock_symbol}_kurs_trends.png", dpi=300)
    plt.close()
    print("Trend-Diagramm als '..._kurs_trends.png' gespeichert.")

### Funktion zum Plotten der Risikoanalyse ###
Funktion XXX

In [9]:
def plot_risk_analysis(stock_data, stock_symbol):
    """
    Visualisiert und vergleicht die historische 21-Tage-Volatilität 
    mit der täglichen innertägigen Handelsspanne.
    """
    print(f"\nGeneriere Risiko-Diagramm für {stock_symbol}...")
    
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.figure(figsize=(12, 6))
    
    # Da Volatilität in Prozent angegeben wird, multiplizieren wir die Intraday-Vola mit 100 für die gleiche Skala
    intraday_pct = stock_data['Intraday_Volatility_Pct'] * 100
    vola_21d_pct = stock_data['Rolling_Volatility_21d'] * 100
    
    # 1. Rollierende Volatilität plotten
    plt.plot(stock_data['Date'], vola_21d_pct, label='21-Tage-Volatilität (Interday, annualisiert)', color='#d62728', linewidth=1.5)
    
    # 2. Innertägige Volatilität plotten
    plt.plot(stock_data['Date'], intraday_pct, label='Innertägige Handelsspanne (Intraday)', color='#9467bd', alpha=0.4, linewidth=0.8)

    # Hier wandert die Legende nach rechts oben ('upper right'):
    apply_standard_chart_styling(
        title=f"Statistischer Risikovergleich ({stock_symbol})", 
        ylabel="Schwankungsbreite in %", 
        legend_loc='upper right'
    )
    
    # Grafik speichern
    plt.savefig(f"{stock_symbol}_risikoanalyse.png", dpi=300)
    plt.close()
    print("Risiko-Diagramm als '..._risikoanalyse.png' gespeichert.")

## Definieren einer Funktion für standardisiertes Chart-Styling ##
Funktion XXX

In [10]:
def apply_standard_chart_styling(title, ylabel, legend_loc='upper left', show_grid=False):
    """
    Wendet ein einheitliches Styling auf das aktuelle Matplotlib-Diagramm an.
    Vermeidet Codedopplungen in den einzelnen Plot-Funktionen.
    """
    plt.title(title, fontsize=13, fontweight='bold', pad=15)
    plt.xlabel("Datum", fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    
    # Die Legende wird flexibel platziert, je nach Diagramm-Typ
    plt.legend(loc=legend_loc, frameon=True, facecolor='white', edgecolor='none')
    
    # Gitter optional zuschalten (manche Diagramme nutzen kein Grid)
    if show_grid:
        plt.grid(True, linestyle=':', alpha=0.6)
        
    plt.tight_layout()

## Definieren (& Aufrufen) der Hauptfunktion ##
Hauptfunktion zur Interaktion mit dem Benutzer, zum Abrufen von Aktiendaten und zu deren Anzeige oder Speicherung in einem lesbaren Format.

In [11]:
def main():
    
    # Interaktive Benutzereingabe
    stock_symbol = input("Gib das Aktienkürzel ein (z.B. 'TSLA'): ").strip()
    start_date = input("Gib das Startdatum ein (YYYY-MM-DD, z.B. '2021-01-01'): ").strip()
    end_date = datetime.now().strftime("%Y-%m-%d")
    
    print(f"\nStarte Analyseprozess für {stock_symbol}...")
    print("--------------------------------------------------")
    
    # Daten laden
    stock_data = get_stock_data(ticker_symbol=stock_symbol, start=start_date, end=end_date)

    if stock_data is None:
        print("Prozess abgebrochen: Datenabruf fehlgeschlagen.")
        return
    
    # Daten bereinigen
    cleaned_stock_data = clean_stock_data(stock_data)

    if cleaned_stock_data is None:
        print("Prozess abgebrochen: Datenbereinigung fehlgeschlagen.")
        return

   # Validierung der Datenstruktur nach Laden der Daten
    cleaned_stock_data.info()

    # Renditen berechnen
    cleaned_stock_data = calculate_metrics(stock_data=cleaned_stock_data)
    
    # Risikoanalyse durchführen
    cleaned_stock_data = calculate_advanced_metrics(stock_data=cleaned_stock_data)
    
    if cleaned_stock_data is None:
        print("Prozess abgebrochen: Die Renditenberechnung und/oder Risikoanalyse ist fehlgeschlagen.")
        return

    # Anzeige-Optionen einstellen
    pd.set_option("display.max_rows", 15)
    pd.set_option("display.max_columns", None)

    # Ausgeben der ersten Zeilen des Dataframes für bessere Übersichtlichkeit/Benutzerfreundlichkeit
    print("\n=== Vorschau der bereinigten Finanzdaten ===")
    print(cleaned_stock_data.tail(5).to_string(index=False))
    print("===========================================================\n")

    print("\n=== Deskriptive Statistik der Finanzdaten ===")
    # mit relevanten Werten bzw. Spalten statistische Kennzahlen berechnen
    stats_cols = ['Close', 'Daily_Return', 'Rolling_Volatility_21d', 'Intraday_Volatility_Pct']
    print(cleaned_stock_data[stats_cols].describe().to_string())
    print("================================================\n")

    # Diagramme generieren
    plot_price_trends(stock_data=cleaned_stock_data, stock_symbol=stock_symbol)
    plot_risk_analysis(stock_data=cleaned_stock_data, stock_symbol=stock_symbol)
    plot_price_history(stock_data=cleaned_stock_data, stock_symbol=stock_symbol)
    plot_daily_returns(stock_data=cleaned_stock_data, stock_symbol=stock_symbol)

    # CSV Export
    file_name = f"{stock_symbol.replace('.', '-')}_analysiert_{end_date}.csv"
    cleaned_stock_data.to_csv(file_name, index=False)
    print(f"Daten erfolgreich gespeichert: '{file_name}'")

main()

Gib das Aktienkürzel ein (z.B. 'TSLA'):  TSLA
Gib das Startdatum ein (YYYY-MM-DD, z.B. '2021-01-01'):  2021-01-01



Starte Analyseprozess für TSLA...
--------------------------------------------------
Lade Daten für TSLA von 2021-01-01 bis 2026-07-08...


[*********************100%***********************]  1 of 1 completed



Daten erfolgreich geladen und bereinigt!

<class 'pandas.DataFrame'>
RangeIndex: 1382 entries, 0 to 1381
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype        
---  ------  --------------  -----        
 0   Date    1382 non-null   datetime64[s]
 1   Open    1382 non-null   float64      
 2   High    1382 non-null   float64      
 3   Low     1382 non-null   float64      
 4   Close   1382 non-null   float64      
 5   Volume  1382 non-null   int64        
dtypes: datetime64[s](1), float64(4), int64(1)
memory usage: 64.9 KB

Berechne tägliche Renditen und absolute Gewinne/Verluste...

Renditen und innertägige Kennzahlen erfolgreich berechnet!

Starte erweiterte Finanzanalyse...

Erweiterte Finanzanalyse erfolgreich abgeschlossen!

=== Vorschau der bereinigten Finanzdaten ===
      Date       Open       High        Low      Close   Volume  Daily_Return  Daily_Gain_Loss_USD  Intraday_Volatility_Pct  Rolling_Volatility_21d     SMA_50   SMA_200
2026-06-30 406.000000 42